In [1]:
import pandas as pd
import os
from tqdm import tqdm
import numpy as np
from scipy.signal import butter, filtfilt, resample_poly, decimate
import matplotlib.pyplot as plt

In [2]:
def bandpass_filter(data, lowcut, highcut, fs, order=4):
    b, a = butter(order, [lowcut, highcut], btype='band', fs=fs)
    y = filtfilt(b, a, data)
    return y

def lowpass_filter(data, cutoff, fs, order=4):
    """
    低通濾波器，使用 Butterworth 設計

    參數:
        data   : ndarray，一維或多維訊號
        cutoff : float，截止頻率（Hz）
        fs     : float，取樣頻率（Hz）
        order  : int，濾波器階數（預設 4）

    回傳:
        y : ndarray，經過低通濾波後的訊號
    """
    nyq = 0.5 * fs  # 奈奎斯特頻率
    normal_cutoff = cutoff / nyq  # 正規化截止頻率
    b, a = butter(order, normal_cutoff, btype='low')
    y = filtfilt(b, a, data)
    return y

In [3]:
file_path = r"F:\M143020071\raw_data_result\conversion_data\healthy/" # 更改病人或健康人
hp_path = r"F:\M143020071\raw_data_result/"
save_path = r"F:\M143020071\raw_data_result\SKNA_signal\ch1\sr2500_500_1000\healthy/" # 更改病人或健康人

In [4]:
if not os.path.exists(save_path):
    os.makedirs(save_path)
    print(f'create directory: {save_path}')
else:
    print(f'{save_path} already exists')

create directory: F:\M143020071\raw_data_result\SKNA_signal\ch1\sr2500_500_1000\healthy/


In [5]:
sampling_rate = 10000  # Hz
down_sampling_rate = 2500  # Hz
# 濾波器設定
f_low = 500  # Hz
f_high = 1000  # Hz
cutoff = 2500  # Hz
order = 6  # 濾波器階數
# 視窗設定
window_size = down_sampling_rate * 20
step_size = down_sampling_rate * 2
# resample 設定
up = 3
down = 5

mode = 0 # change
resample = 0
filter_type = 1
"""
mode = 0 ---> healthy
mode = 1 ---> patient
resample = 0 ---> downsample
resample = 1 ---> resample
filter_type = 1 ---> bandpass filter
filter_type = 2 ---> lowpass filter
"""

'\nmode = 0 ---> healthy\nmode = 1 ---> patient\nresample = 0 ---> downsample\nresample = 1 ---> resample\nfilter_type = 1 ---> bandpass filter\nfilter_type = 2 ---> lowpass filter\n'

In [6]:
down_sample_gap = int(sampling_rate / down_sampling_rate)
print(f'down sample gap: {down_sample_gap}')

down sample gap: 4


In [7]:
hp = pd.read_csv(hp_path + "MACE_h_id.csv")  # 更改為健康人或病人
hp_raw = hp.iloc[:, 0].tolist()
hp_id = hp.iloc[:, 1].tolist()

In [8]:
too_short = []
for i in tqdm(range(hp.shape[0]), desc="Processing files"): # hp.shape[0]
    try:
        # print(f"{hp_raw[i]}.csv")
        data = pd.read_csv(file_path + f"{hp_raw[i]}.csv")
        ch_1 = data.iloc[:, 1].values
        # lead_1_filtered = None

        if filter_type == 1:
            print('bandpass filter')
            lead_1_filtered = bandpass_filter(ch_1, f_low, f_high, sampling_rate, order)
        elif filter_type == 2:
            print('lowpass filter')
            lead_1_filtered = lowpass_filter(ch_1, cutoff, sampling_rate, order)
        else:
            raise ValueError("filter_type error, must be 1 or 2")

        if len(lead_1_filtered) >= sampling_rate * 60 * 5: # 至少5分鐘
            if len(lead_1_filtered) >= sampling_rate * 60 * 7: # 至少7分鐘
                print(f'{hp_raw[i]} signal greater than 7 minutes, extract 2-7 minutes')
                lead_1_filtered = lead_1_filtered[sampling_rate * 60 * 2 :sampling_rate * 60 * 7]

                if resample == 0:
                    lead_1_downsampled = decimate(lead_1_filtered, down_sample_gap)
                    print(f'downsampled signal length: {len(lead_1_downsampled)}')
                elif resample == 1:
                    lead_1_downsampled = resample_poly(lead_1_filtered, up, down)
                    print(f'resampled signal length: {len(lead_1_downsampled)}')
                else:
                    raise ValueError("resample error, must be 0 or 1")

                win_num = int(((len(lead_1_downsampled)-window_size)/step_size)+1)

                all_signal = []
                mean = []
                std = []
                for w in range(win_num):
                    st = step_size * w
                    ed = step_size * w + window_size
                    window = lead_1_downsampled[st:ed]
                    mean_value = np.mean(window)
                    std_value = np.std(window)
                    all_signal.append(window)
                    mean.append(mean_value)
                    std.append(std_value)

                mu = np.median(mean)
                sigma = np.median(std)

                signal_arr = np.vstack(all_signal)  # shape: (n_windows, window_length)
                signal_arr_std = (signal_arr - mu) / sigma  # 用 mu, sigma 做全體標準化

                if mode == 0:
                    label_arr = np.zeros((len(all_signal), 1))
                    signal_std_arr = np.concatenate([label_arr, signal_arr_std], axis=1)  # multirocket 要求 float64
                    np.save(save_path + f"{hp_id[i]}.npy", signal_std_arr)
                elif mode == 1:
                    label_arr = np.ones((len(all_signal), 1))
                    signal_std_arr = np.concatenate([label_arr, signal_arr_std], axis=1)  # multirocket 要求 float64
                    np.save(save_path + f"{hp_id[i]}.npy", signal_std_arr)
                else:
                    raise ValueError("mode error, must be 0 or 1")

            elif len(lead_1_filtered) < sampling_rate * 60 * 7:
                print(f'{hp_raw[i]} signal less than 7 minutes, extract the last 5 minutes ')
                lead_1_filtered = lead_1_filtered[-sampling_rate * 60 * 5 :]
                

                if resample == 0:
                    lead_1_downsampled = lead_1_filtered[::down_sample_gap]
                    print(f'downsampled signal length: {len(lead_1_downsampled)}')
                elif resample == 1:
                    lead_1_downsampled = resample_poly(lead_1_filtered, up, down)
                    print(f'resampled signal length: {len(lead_1_downsampled)}')
                else:
                    raise ValueError("resample error, must be 0 or 1")

                win_num = int(((len(lead_1_downsampled) - window_size) / step_size) + 1)

                all_signal = []
                mean = []
                std = []
                for w in range(win_num):
                    st = step_size * w
                    ed = step_size * w + window_size
                    window = lead_1_downsampled[st:ed]
                    mean_value = np.mean(window)
                    std_value = np.std(window)
                    all_signal.append(window)
                    mean.append(mean_value)
                    std.append(std_value)

                mu = np.median(mean)
                sigma = np.median(std)

                signal_arr = np.vstack(all_signal)  # shape: (n_windows, window_length)
                signal_arr_std = (signal_arr - mu) / sigma  # 用 mu, sigma 做全體標準化

                if mode == 0:
                    label_arr = np.zeros((len(all_signal), 1))
                    signal_std_arr = np.concatenate([label_arr, signal_arr_std], axis=1)  # multirocket 要求 float64
                    np.save(save_path + f"{hp_id[i]}.npy", signal_std_arr)
                elif mode == 1:
                    label_arr = np.ones((len(all_signal), 1))
                    signal_std_arr = np.concatenate([label_arr, signal_arr_std], axis=1)  # multirocket 要求 float64
                    np.save(save_path + f"{hp_id[i]}.npy", signal_std_arr)
                else:
                    raise ValueError("mode error, must be 0 or 1")

        else:
            print(f'{hp_raw[i]} signal less than 5 minutes, extract the last 5 minutes ')
            too_short.append(hp_raw[i])
            pass

    except Exception as e:
        print(f"Error processing file {hp_raw[i]}: {e}")
        continue

    # 清除記憶體
    del data

Processing files:   0%|          | 0/434 [00:00<?, ?it/s]

bandpass filter


Processing files:   0%|          | 1/434 [00:02<19:54,  2.76s/it]

414490914181 signal greater than 7 minutes, extract 2-7 minutes
downsampled signal length: 750000
bandpass filter
534491345995 signal greater than 7 minutes, extract 2-7 minutes


Processing files:   0%|          | 2/434 [00:04<17:25,  2.42s/it]

...

downsampled signal length: 750000
bandpass filter
454552424829 signal greater than 7 minutes, extract 2-7 minutes


Processing files: 100%|█████████▉| 433/434 [17:24<00:02,  2.23s/it]

downsampled signal length: 750000
bandpass filter
374552404823 signal greater than 7 minutes, extract 2-7 minutes


Processing files: 100%|██████████| 434/434 [17:26<00:00,  2.41s/it]

downsampled signal length: 750000


In [9]:
if not too_short:
    print(f'without too short signal')
else:
    print(f'too short signal: {too_short}')
    too_short_df = pd.DataFrame(too_short, columns=['ID'])
    too_short_df.to_csv(save_path + "too_short.csv", index=False)

without too short signal
